In [ ]:
#模型训练Autoencoder自编码结构：输入层，编码层，解码层，输出层


In [3]:
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# 特征文件目录
save_dir = Path("/home/jovyan/MIMII-Project/features")

# 读取正常和异常特征
normal_features = np.load(save_dir / "normal_features.npy")
abnormal_features = np.load(save_dir / "abnormal_features.npy")

print("正常特征：", normal_features.shape)
print("异常特征：", abnormal_features.shape)
print("PyTorch版本：", torch.__version__)

正常特征： (1011, 313, 128)
异常特征： (407, 313, 128)
PyTorch版本： 2.6.0+cu124


In [4]:
# 正常音频划分为训练集和验证集
#正常音频学习一部分验证一部分 异常音频和正常音频一起测试
#导入划分工具
from sklearn.model_selection import train_test_split
normal_train, normal_temp = train_test_split(
    normal_features,
    test_size=0.2,#拿出20%作为暂存数据 剩余80%训练
    random_state=42#划分固定每次使用相同种子 
)
normal_val,normal_test = train_test_split(
    normal_temp,
    test_size=0.5,
    random_state=42#得到测试集 验证集 训练集
)

# 异常音频全部作为测试集
abnormal_test = abnormal_features

print("训练集：", normal_train.shape)
print("验证集：", normal_val.shape)
print("异常测试集：", abnormal_test.shape)
print("正常测试集：",normal_test.shape)

训练集： (808, 313, 128)
验证集： (101, 313, 128)
异常测试集： (407, 313, 128)
正常测试集： (102, 313, 128)


In [5]:
class Autoencoder(nn.Module):#括号表示继承关系
    def __init__(self):#初始化
        super().__init__()
        self.encoder = nn.Sequential(#保存编码器网络
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,16),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(#解码器
            nn.Linear(16,64),
            nn.ReLU(),
            nn.Linear(64,128),
        )
    def forward(self,x):#定义数据流通方向
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
#测试模型能否正常运行
model = Autoencoder()#创建对象
import torch
x = torch.randn(32,128)
output = model(x)
print("输入形状：",x.shape)
print("输出形状:",model(x).shape)



输入形状： torch.Size([32, 128])
输出形状: torch.Size([32, 128])


In [18]:
#整理训练数据 模型只能接受两维 要将808段音频里每一帧都展开
print(type(normal_train))
print(normal_train.shape)

<class 'numpy.ndarray'>
(808, 313, 128)


In [6]:
normal_train = normal_train.reshape(252904,128)
print (normal_train.shape)

(252904, 128)


In [7]:
normal_train = torch.tensor(normal_train, dtype=torch.float32)#tensor是专门的张量储存并计算数据
print(type(normal_train))
print(normal_train.shape)

<class 'torch.Tensor'>
torch.Size([252904, 128])


In [8]:
normal_val = normal_val.reshape(101*313,128)
print(normal_val.shape)

(31613, 128)


In [10]:
normal_val = torch.tensor(normal_val,dtype=torch.float32)
print(type(normal_val))
print(normal_val.shape)

<class 'torch.Tensor'>
torch.Size([31613, 128])


In [11]:
#数据加载器DataLoader
from torch.utils.data import TensorDataset, DataLoader
#TensorDataset 把 Tensor 数据组织成数据集
#DataLoader 按批次读取数据
train_dataset = TensorDataset(normal_train)#将normal_train创建成pytorch的数据集
train_loader = DataLoader(
    train_dataset,
    batch_size=256,#每次读取256条
    shuffle=True#每轮训练前打乱数据集
)
print(len(train_loader))
data = next(iter(train_loader))#创建可以取出数据的工具
print(data[0].shape)#保存取出的数据

988
torch.Size([256, 128])


In [12]:
#定义损失函数和优化器
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        data = batch[0]

        # 1. 清空梯度
        optimizer.zero_grad()

        # 2. 前向传播
        outputs = model(data)

        # 3. 计算重建误差
        loss = criterion(outputs, data)

        # 4. 反向传播
        loss.backward()

        # 5. 更新模型参数
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}], "
        f"Loss: {avg_loss:.6f}"
    )

Epoch [1/20], Loss: 35.048823
Epoch [2/20], Loss: 8.817410
Epoch [3/20], Loss: 7.817360
Epoch [4/20], Loss: 7.474422
Epoch [5/20], Loss: 6.973951
Epoch [6/20], Loss: 6.587389
Epoch [7/20], Loss: 6.454919
Epoch [8/20], Loss: 6.350086
Epoch [9/20], Loss: 6.232153
Epoch [10/20], Loss: 6.193180
Epoch [11/20], Loss: 6.170220
Epoch [12/20], Loss: 6.148021
Epoch [13/20], Loss: 6.127161
Epoch [14/20], Loss: 6.114632
Epoch [15/20], Loss: 6.103294
Epoch [16/20], Loss: 6.094292
Epoch [17/20], Loss: 6.088889
Epoch [18/20], Loss: 6.082015
Epoch [19/20], Loss: 6.076832
Epoch [20/20], Loss: 6.070450


In [21]:
#训练模型
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        data = batch[0]

        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, data)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.6f}")

Epoch [1/20], Loss: 5.517619
Epoch [2/20], Loss: 5.514480
Epoch [3/20], Loss: 5.513998
Epoch [4/20], Loss: 5.510641
Epoch [5/20], Loss: 5.510510
Epoch [6/20], Loss: 5.507879
Epoch [7/20], Loss: 5.506726
Epoch [8/20], Loss: 5.505346
Epoch [9/20], Loss: 5.503374
Epoch [10/20], Loss: 5.502584
Epoch [11/20], Loss: 5.500900
Epoch [12/20], Loss: 5.500333
Epoch [13/20], Loss: 5.499416
Epoch [14/20], Loss: 5.496533
Epoch [15/20], Loss: 5.496247
Epoch [16/20], Loss: 5.495042
Epoch [17/20], Loss: 5.493620
Epoch [18/20], Loss: 5.493310
Epoch [19/20], Loss: 5.491317
Epoch [20/20], Loss: 5.488756


In [16]:
#检验评估集
val_dataset = TensorDataset(normal_val)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False
)
model.eval()

val_loss = 0

with torch.no_grad():
    for batch in val_loader:
        data = batch[0]
        outputs = model(data)
        loss = criterion(outputs, data)
        val_loss += loss.item()

avg_val_loss = val_loss / len(val_loader)

print("Validation Loss:", avg_val_loss)

Validation Loss: 5.756340026855469


In [32]:
#让模型判断声音是否正确
normal_test:   (102, 313, 128)
abnormal_test: (407, 313, 128)
def get_anomaly_scores(model, features):
    model.eval()
    scores = []

    with torch.no_grad():
        for audio in features:
            audio = torch.as_tensor(
                audio,
                dtype=torch.float32
            )

            outputs = model(audio)

            error = torch.mean((outputs - audio) ** 2)
            scores.append(error.item())

    return np.array(scores)
normal_scores = get_anomaly_scores(model, normal_test)

print("正常测试集分数：", normal_scores.shape)
print("正常平均异常分数：", normal_scores.mean())
abnormal_scores = get_anomaly_scores(model, abnormal_test)

print("异常测试集分数：", abnormal_scores.shape)
print("异常平均分数：", abnormal_scores.mean())
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

正常测试集分数： (102,)
正常平均异常分数： 5.760873948826509
异常测试集分数： (407,)
异常平均分数： 6.176795895035203


In [33]:
y_true = np.concatenate([
    np.zeros(len(normal_scores)),
    np.ones(len(abnormal_scores))
])

y_scores = np.concatenate([
    normal_scores,
    abnormal_scores
])

# ROC-AUC：衡量区分正常和异常声音的能力
auc = roc_auc_score(y_true, y_scores)

# 用正常验证集分数的95%分位数作为阈值
val_scores = get_anomaly_scores(
    model,
    normal_val.reshape(101, 313, 128)
)
threshold = np.percentile(val_scores, 95)
# 根据阈值判断正常或异常
y_pred = (y_scores > threshold).astype(int)

print("ROC-AUC:", auc)
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))
print("Threshold:", threshold)
from pathlib import Path

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

torch.save(
    model.state_dict(),
    model_dir / "mimii_autoencoder.pkl"
)

print("模型保存成功！")

ROC-AUC: 0.611865876571759
Precision: 0.8620689655172413
Recall: 0.06142506142506143
F1: 0.11467889908256881
Threshold: 8.703571319580078
模型保存成功！


In [34]:
print("正常声音平均分数：", normal_scores.mean())
print("异常声音平均分数：", abnormal_scores.mean())

print("正常声音分数95%分位数：", np.percentile(normal_scores, 95))
print("异常声音分数95%分位数：", np.percentile(abnormal_scores, 95))

print("当前阈值：", threshold)

正常声音平均分数： 5.760873948826509
异常声音平均分数： 6.176795895035203
正常声音分数95%分位数： 8.3955442905426
异常声音分数95%分位数： 9.188390350341797
当前阈值： 8.703571319580078


In [35]:
import torch
from pathlib import Path

model_path = Path("models/mimii_autoencoder.pkl")

state_dict = torch.load(
    model_path,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(state_dict)

print("模型权重加载成功！")

模型权重加载成功！
